In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df = pd.read_csv("../data/processed/rossmann_base.csv", parse_dates=["Date"])
df.shape


C:\Users\TestAdmin\AppData\Local\Temp\ipykernel_25092\2854638647.py:7: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/rossmann_base.csv", parse_dates=["Date"])


(844392, 22)

In [6]:
drop_cols = [
    "Date",        # temporal index, not a predictor directly
    "Customers"   # data leakage
]

df_fe = df.drop(columns=drop_cols)
df_fe.columns

Index(['Store', 'DayOfWeek', 'Sales', 'Open', 'Promo', 'StateHoliday',
       'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2',
       'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month',
       'Week', 'Day'],
      dtype='object')

In [7]:
X = df_fe.drop(columns=["Sales"])
y = df_fe["Sales"]

X.shape, y.shape

((844392, 19), (844392,))

In [8]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_cols, numerical_cols

(['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval'],
 ['Store',
  'DayOfWeek',
  'Open',
  'Promo',
  'SchoolHoliday',
  'CompetitionDistance',
  'CompetitionOpenSinceMonth',
  'CompetitionOpenSinceYear',
  'Promo2',
  'Promo2SinceWeek',
  'Promo2SinceYear',
  'Year',
  'Month',
  'Week',
  'Day'])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

X_train.shape, X_test.shape

((675513, 19), (168879, 19))

In [10]:
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)

# Align test set to train set columns
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

In [11]:
num_cols_enc = X_train_enc.select_dtypes(include=["int64", "float64"]).columns

imputer = SimpleImputer(strategy="median")

X_train_enc[num_cols_enc] = imputer.fit_transform(X_train_enc[num_cols_enc])
X_test_enc[num_cols_enc] = imputer.transform(X_test_enc[num_cols_enc])

In [12]:
X_train_enc.isnull().sum().sum(), X_test_enc.isnull().sum().sum()

(np.int64(0), np.int64(0))

In [13]:
X_train_enc.to_csv("../data/processed/X_train.csv", index=False)
X_test_enc.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)